In [1]:
from hana_ml import dataframe
url, port, user, pwd = "810070ba-df1a-4553-9a82-23690dc0158e.hana.demo-hc-3-haas-hc-dev.dev-aws.hanacloud.ondemand.com", \
443, "SAPUSER", "nIL0yr8S"
cc = dataframe.ConnectionContext(url, port, user, pwd)

In [2]:
import numpy as np
np.random.seed(2023)
x = np.arange(256) / 16 
y = np.array(list(2 * np.sin(x[:96] * np.pi) ) + list(2 * np.sin(x[96:] * 2 * np.pi)))
z = np.array(list(x[:128] / 2) + list(8 - x[128:] / 2)) 

In [3]:
import pandas as pd
cpd_data = pd.DataFrame(dict(ID=range(256),
                             DATE=pd.date_range('2022-02-02', periods=256),
                             VAL=y+z+0.1*(np.random.rand(256) - 0.5)))

In [4]:
from hana_ml.dataframe import create_dataframe_from_pandas
bcpd_df = create_dataframe_from_pandas(cc, cpd_data,
                                      "BCPD_SIM_DATA_TBL",
                                      force=True)

100%|██████████| 1/1 [00:00<00:00,  2.69it/s]


In [5]:
from hana_ai.tools.hana_ml_tools.change_point_tools import BayesianChangePoint
cpd_tool = BayesianChangePoint(cc)

In [6]:
from hana_ml.algorithms.pal.tsa.changepoint import BCPD
bcpd = BCPD(max_tcp=1, max_scp=1, max_iter=1000)
res = bcpd.fit_predict(cc.table('BCPD_SIM_DATA_TBL'), key='DATE', endog='VAL')

In [7]:
input_bcpd = dict(table_name="BCPD_SIM_DATA_TBL",
                  key='DATE', endog='VAL',
                  max_tcp=1, max_scp=1,
                  #max_harmonic_order=3,
                  max_iter=1000)
cpd_tool.run(tool_input=input_bcpd)

'{"trend_change_points": "2022-05-09 00:00:00", "seasonal_change_points": "", "periods": "27", "bcpd_decomposed_table": "BCPD_SIM_DATA_TBL_BCPD_DECOMPOSED"}'

In [8]:
from langchain.agents import initialize_agent, AgentType
from gen_ai_hub.proxy.langchain import init_llm
llm = init_llm('gpt-4o', temperature=0.0, max_tokens=2000) # used to do logical reasoning
tools = [cpd_tool] # Add any tools here
agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)

C:\Users\I326292\AppData\Local\Temp\ipykernel_8072\2966300920.py:5: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)


In [9]:
instruction = "Please apply Bayesian change point detection to the time-series data in table BCPD_SIM_DATA_TBL," +\
              "where key is DATE and endog is VAL, using maximally 1000 iterations"
agent_chain.invoke(instruction)

{'input': 'Please apply Bayesian change point detection to the time-series data in table BCPD_SIM_DATA_TBL,where key is DATE and endog is VAL, using maximally 1000 iterations',
 'output': 'To apply Bayesian change point detection, I need additional information regarding the maximum number of trend change points (max_tcp) and season change points (max_scp) you would like to detect. Could you please provide these details?'}

In [10]:
instruction = "Please apply Bayesian change point detection to the time-series data in table BCPD_SIM_DATA_TBL using maximally 1000 iterations," +\
"where key is DATE, endog is VAL, max_tcp is 1 and max_scp is 1."
agent_chain.invoke(instruction)

{'input': 'Please apply Bayesian change point detection to the time-series data in table BCPD_SIM_DATA_TBL using maximally 1000 iterations,where key is DATE, endog is VAL, max_tcp is 1 and max_scp is 1.',
 'output': '{"trend_change_points": "2022-04-20 00:00:00", "seasonal_change_points": "", "periods": "127", "bcpd_decomposed_table": "BCPD_SIM_DATA_TBL_BCPD_DECOMPOSED"}'}

In [11]:
cc.drop_table("BCPD_SIM_DATA_TBL_BCPD_DECOMPOSED")
cc.drop_table("BCPD_SIM_DATA_TBL")
cc.close()